In [ ]:
import pandas as pd 
import numpy as np 
import re
from tqdm import tqdm
import matplotlib.pyplot as plt
from datetime import datetime
import geopandas as gpd
from scipy.optimize import curve_fit
from sklearn.linear_model import LinearRegression

import os
from IPython.display import clear_output
%run ./methods/methods.py
clear_output()

### Hots-spots analysis


In [ ]:
date = datetime.today().strftime('%Y%m%d')

In [ ]:
directory_path = './data/outputs/'
for file in os.listdir(directory_path):
    if 'wstreetInfos' in file:
        filepath = os.path.join(directory_path, file)

df = pd.read_csv(filepath,sep=";",usecols = ['adr_init','ville_init','cp_init',
                                                        'adr_geo', 'ville_geo','cp_geo','y','x',
                                                        'not_matched_brute','not_matched_geo','contains_street_type'])


df_odonyme = pd.read_csv("./data/odonymes.txt",delimiter="|")
liste_odonyme = pd.Series(df_odonyme['synonym'].str.upper())
clear_output()

In [ ]:
df['contains_street_type'] = df['contains_street_type'].astype(bool)

df_w_street = df[df['contains_street_type']==True]
df_wout_street = df[df['contains_street_type']==False]

#### 1. For addresses containing a road type

a. Group addresses by address, town and cpville

In [ ]:
df_counts = (
    df_w_street
    .groupby(by=['adr_init', 'ville_init', 'cp_init'])
    .size()
    .reset_index(name='count')
)
df_counts.sort_values(by="count",ascending=False)

df_candidats = df_w_street.merge(df_counts, on=['adr_init', 'ville_init', 'cp_init'])

In [ ]:
cp_chu = 75015
df_candidats = df_candidats[(df_candidats['cp_init']!=cp_chu)&(df_candidats['count']>=30)]


plt.hist(df_candidats['count'], bins=50, edgecolor='black')
plt.title('Distribution of Grouped Counts')
plt.xlabel('Count')
plt.ylabel('Frequency')
plt.show()

In [ ]:
gdf_cand = gpd.GeoDataFrame(
    df_candidats, geometry=gpd.points_from_xy(df_candidats['x'], df_candidats['y']),crs="EPSG:4326")
gdf_cand = gdf_cand.to_crs("EPSG:2154")


c. Compute distance from Hospital center

In [ ]:
from shapely.geometry import Point

## Point corresponding to HEGP hospital 
### TO REPLACE WITH OWN CENTER 
latitude = 48.83931098867617
longitude = 2.2753718727661623

point_hegp = gpd.GeoDataFrame(
    {"geometry": [Point(longitude, latitude)]},  
    crs="EPSG:4326" 
)
# Reproject to Lambert93 (EPSG:2154)
point_hegp = point_hegp.to_crs("EPSG:2154")


gdf_cand["distance_m"] = calculer_distance_euclidienne(gdf_cand.geometry.x.values, gdf_cand.geometry.y.values, point_hegp['geometry'].x.values, point_hegp['geometry'].y.values)

gdf_cand["distance_km"] = gdf_cand['distance_m'] / 1000

gdf_cand = gdf_cand[gdf_cand['distance_km']>=1]




d. inverse regression of number by distance

In [ ]:
### Visualisation de la distribution 

plt.scatter(x=gdf_cand['distance_km'],y= gdf_cand['count']) #bins=50, edgecolor='black'
plt.xlabel('Distance')
plt.ylabel('count')
plt.show()

In [ ]:
gdf_candidates = gdf_cand[(gdf_cand['distance_km'] > 0) & (gdf_cand['count'] > 0)]

min_distance = gdf_candidates["distance_km"].min()
max_distance = gdf_candidates["distance_km"].max()

bin_edges = np.arange(0, max_distance + 1, 1)  
gdf_candidates["distance_bin"] = pd.cut(gdf_candidates["distance_km"], bins=bin_edges)


upper_contour = gdf_candidates.groupby('distance_bin').apply(lambda group: group.nlargest(1, 'count')).reset_index(drop=True)

def inverse_func(x, a):
    return a / x

popt, _ = curve_fit(inverse_func, upper_contour['distance_km'], upper_contour['count'])
a_estimated = popt[0]

x_vals = np.linspace(gdf_candidates['distance_km'].min(), gdf_candidates['distance_km'].max(), 500)
y_vals = inverse_func(x_vals, a_estimated)


plt.scatter(gdf_candidates['distance_km'], gdf_candidates['count'], label='Data Points', alpha=0.5)
plt.scatter(upper_contour['distance_km'], upper_contour['count'], color='orange', label='Upper Contour Points')
plt.plot(x_vals, y_vals, color='red', label=f'Fit: y = {a_estimated:.2f}/x')
plt.xlabel('Distance (km)')
plt.ylabel('Count')
plt.title('Inverse Function Fit to Upper Contour')
plt.legend()
# plt.xlim(0, 700)
# plt.ylim(0, 700)
plt.grid(True)
plt.show()

In [ ]:
# Calculate the predicted values from the fitted inverse function
gdf_candidates['predicted_count'] = inverse_func(gdf_candidates['distance_km'], a_estimated)

# Filter points where actual count is higher than the predicted count
points_above_curve = gdf_candidates[gdf_candidates['count'] > gdf_candidates['predicted_count']]

threshold = 0.80 ## point who are less than 20% of the predicted count 
points_below_threshold = gdf_candidates[(gdf_candidates['count'] > threshold* gdf_candidates['predicted_count']) & (gdf_candidates['count'] < gdf_candidates['predicted_count'])]


# Plot these points along with the fitted curve
plt.scatter(gdf_candidates['distance_km'], gdf_candidates['count'], label='All Data Points', alpha=0.5)
plt.scatter(points_below_threshold['distance_km'], points_below_threshold['count'], color='yellow', label='Points 20% Below Curve', marker='o')
plt.scatter(points_above_curve['distance_km'], points_above_curve['count'], color='green', label='Points Above Curve', marker='o')
plt.plot(x_vals, y_vals, color='red', label=f'Fit: y = {a_estimated:.2f}/x')

# Add axis labels, title, and legend
plt.xlabel('Distance (km)')
plt.ylabel('Count')
plt.title('Points Above the Fitted Curve')
plt.legend()
plt.xlim(0, gdf_candidates['distance_km'].max() + 5)
plt.ylim(0, gdf_candidates['count'].max() + 20)
plt.grid(True)
plt.show()


e. manual verification of identified addresses

In [ ]:
filtered_df = gdf_candidates[(gdf_candidates['count'] > gdf_candidates['predicted_count']) | (gdf_candidates['count'] > threshold* gdf_candidates['predicted_count'])]

In [ ]:
adr_to_check = filtered_df['adr_init'].tolist()

adr_candidates = ['A REMPLIR AVEC ADRESSES CANDIDATES IDENTIFIEES ']

print(len(adr_to_check))

print(len(adr_candidates))